# Ridge Regression Volatility Backtest

CPU-only walk-forward backtest with rolling robust scaling and HAR features.

In [ ]:
import os
import subprocess
import sys

# Clone repo if not present
if not os.path.exists("harxhar"):
    subprocess.run(["git", "clone", "https://github.com/your-org/harxhar.git"], check=True)
os.chdir("harxhar/colab")

# Install dependencies
subprocess.check_call(
    [sys.executable, "-m", "pip", "-q", "install", "numpy", "pandas", "scikit-learn", "pyarrow", "numba", "tqdm"]
)

# Add colab to path
sys.path.insert(0, ".")

In [ ]:
# Parameters
HORIZON = 1
TRAIN_WINDOW_DAYS = 500
DATA_PATH = "all30min"
PERIODS_PER_DAY = 48
TRAIN_WINDOW = TRAIN_WINDOW_DAYS * PERIODS_PER_DAY
print(f"Horizon: {HORIZON}, Train window: {TRAIN_WINDOW_DAYS} days = {TRAIN_WINDOW} periods")

In [ ]:
import numpy as np
import pandas as pd

from src.loading import load_raw_data
from src.transforms import robust_transform

df = load_raw_data(DATA_PATH)
print(f"Loaded {len(df)} rows, columns: {list(df.columns)}")

adj_rv, baseline = robust_transform(df, "RV", is_target=True)
df["adj_RV"] = adj_rv
df["baseline"] = baseline
df[["t", "RV", "adj_RV", "baseline"]].head(10)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from src.transforms import generate_har_features, resolve_har_lags

df, feature_names = generate_har_features(df, target_col="adj_RV")
print(f"HAR lags: {resolve_har_lags()}")
print(f"Feature names: {feature_names}")

# Drop burn-in rows
max_lag = resolve_har_lags()[-1]
df = df.iloc[max_lag:].reset_index(drop=True)

# Correlation heatmap
corr_cols = ["adj_RV"] + feature_names
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", ax=ax, cmap="coolwarm")
ax.set_title("HAR Feature Correlations")
plt.tight_layout()
plt.show()

In [ ]:
from src.transforms import apply_horizon_shift

X = df[feature_names].values.astype(np.float64)
y = df["adj_RV"].values.astype(np.float64)
dates = df["t"]
baselines = df["baseline"].values.astype(np.float64)

X, y, dates, baselines = apply_horizon_shift(X, y, dates, baselines, HORIZON)
print(f"After horizon shift: X {X.shape}, y {y.shape}")
print(f"Date range: {dates.iloc[0]} to {dates.iloc[-1]}")

In [ ]:
from sklearn.linear_model import Ridge

from src.scaling import run_backtest
from src.evaluation import apply_duan_smearing


def model_fn():
    return Ridge(alpha=1.0)


preds = run_backtest(model_fn, X, y, train_win=TRAIN_WINDOW, refit_frequency=1, use_scaling=True)

oos_start = TRAIN_WINDOW
y_oos = y[oos_start:]
dates_oos = dates.iloc[oos_start:].values
baselines_oos = baselines[oos_start:]

pred_raw, true_raw = apply_duan_smearing(preds, y_oos, baselines_oos)

# Plot predictions vs actual
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(dates_oos[-2000:], y_oos[-2000:], alpha=0.7, label="True (adj)")
axes[0].plot(dates_oos[-2000:], preds[-2000:], alpha=0.7, label="Pred (adj)")
axes[0].set_title("Adjusted-scale predictions (last 2000)")
axes[0].legend()
axes[1].plot(dates_oos[-2000:], true_raw[-2000:], alpha=0.7, label="True (raw)")
axes[1].plot(dates_oos[-2000:], pred_raw[-2000:], alpha=0.7, label="Pred (raw)")
axes[1].set_title("Raw-scale predictions (last 2000)")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
from src.evaluation import calculate_metrics

results_df = pd.DataFrame(
    {
        "date": dates_oos,
        "horizon": HORIZON,
        "true_adj": y_oos,
        "pred_adj": preds,
        "true_raw": true_raw,
        "pred_raw": pred_raw,
    }
)

metrics = calculate_metrics(results_df)
print("\n".join(f"{k:>10s}: {v:.6f}" for k, v in metrics.items()))

In [ ]:
%%writefile ../src/ml_ridge.py
"""Ridge regression volatility backtest executor."""

import argparse
import os

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge

from src.loading import load_raw_data
from src.transforms import (
    robust_transform,
    resolve_har_lags,
    generate_har_features,
    apply_horizon_shift,
    PERIODS_PER_DAY,
)
from src.scaling import run_backtest
from src.evaluation import apply_duan_smearing


def main() -> None:
    parser = argparse.ArgumentParser(description="Ridge walk-forward backtest")
    parser.add_argument("--data-path", default="all30min")
    parser.add_argument("--horizon", type=int, default=1)
    parser.add_argument("--train-window", type=int, default=500, help="training window in days")
    parser.add_argument("--start", type=int, default=0)
    parser.add_argument("--end", type=int, default=-1)
    parser.add_argument("--output-file", required=True)
    args = parser.parse_args()

    train_win_periods = args.train_window * PERIODS_PER_DAY

    df = load_raw_data(args.data_path)
    adj_rv, baseline = robust_transform(df, "RV", is_target=True)
    df["adj_RV"] = adj_rv
    df["baseline"] = baseline

    df, feature_names = generate_har_features(df, target_col="adj_RV")
    max_lag = resolve_har_lags()[-1]
    df = df.iloc[max_lag:].reset_index(drop=True)

    X = df[feature_names].values.astype(np.float64)
    y = df["adj_RV"].values.astype(np.float64)
    dates = df["t"]
    baselines = df["baseline"].values.astype(np.float64)

    X, y, dates, baselines = apply_horizon_shift(X, y, dates, baselines, args.horizon)

    start = args.start
    end = len(X) if args.end == -1 else args.end
    X_chunk = X[start:end]
    y_chunk = y[start:end]
    dates_chunk = dates.iloc[start:end].reset_index(drop=True)
    baselines_chunk = baselines[start:end]

    if train_win_periods >= len(X_chunk):
        raise ValueError(f"train_window ({train_win_periods} periods) >= chunk size ({len(X_chunk)})")

    model_fn = lambda: Ridge(alpha=1.0)
    preds = run_backtest(model_fn, X_chunk, y_chunk, train_win=train_win_periods, refit_frequency=1, use_scaling=True)

    oos_start = train_win_periods
    y_oos = y_chunk[oos_start:]
    dates_oos = dates_chunk.iloc[oos_start:].values
    baselines_oos = baselines_chunk[oos_start:]

    pred_raw, true_raw = apply_duan_smearing(preds, y_oos, baselines_oos)

    results = pd.DataFrame({
        "date": dates_oos, "horizon": args.horizon,
        "true_adj": y_oos, "pred_adj": preds,
        "true_raw": true_raw, "pred_raw": pred_raw,
    })

    os.makedirs(os.path.dirname(args.output_file) or ".", exist_ok=True)
    results.to_csv(args.output_file, index=False)
    print(f"Saved {len(results)} rows -> {args.output_file}")


if __name__ == "__main__":
    main()